In [303]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [304]:
df = pd.read_csv('./data/spam_preprocessed.csv', encoding='latin-1')
# vectorizer = TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1, 2))
# vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=3000, min_df=2)

In [305]:
x = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train_vectorized = vectorizer.fit_transform(X_train)
x_test_vectorized = vectorizer.transform(X_test)

In [306]:
print(f"x_train shape: {x_train_vectorized.shape}")
print(f"x_test shape: {x_test_vectorized.shape}")
classifier = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    C=1.0,
    class_weight='balanced',
    random_state=4239
)
model = classifier.fit(x_train_vectorized, y_train)
y_pred = model.predict(x_test_vectorized)

x_train shape: (4456, 3000)
x_test shape: (1114, 3000)


In [307]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9811
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.93      0.93      0.93       149

    accuracy                           0.98      1114
   macro avg       0.96      0.96      0.96      1114
weighted avg       0.98      0.98      0.98      1114

Confusion Matrix:
[[954  11]
 [ 10 139]]


In [308]:
test_spam_data = pd.read_json('./data/test_spam_data.json')
test_spam_data.replace({'label': {'ham': 0, 'spam': 1}}, inplace=True)
test_spam_data['vectorized_message'] = vectorizer.transform(test_spam_data['email_text']).toarray().tolist()
test_spam_data = test_spam_data.sample(frac=1, random_state=1290).reset_index(drop=True)
test_spam_data['label'] = test_spam_data['label'].astype(int)

In [309]:
y_test_labels = test_spam_data['label']
separate_X_test = np.vstack(test_spam_data['vectorized_message'])
random_y_pred = model.predict(separate_X_test)
confidence = model.predict_proba(separate_X_test)

In [310]:
test_accuracy = accuracy_score(y_test_labels, random_y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")
print("Test Classification Report:")
print(classification_report(y_test_labels, random_y_pred))
print("Test Confusion Matrix:")
print(confusion_matrix(y_test_labels, random_y_pred))

Test Accuracy: 0.7647
Test Classification Report:
              precision    recall  f1-score   support

           0       0.56      1.00      0.71         5
           1       1.00      0.67      0.80        12

    accuracy                           0.76        17
   macro avg       0.78      0.83      0.76        17
weighted avg       0.87      0.76      0.77        17

Test Confusion Matrix:
[[5 0]
 [4 8]]


In [311]:
test_spam_data['predicted'] = random_y_pred
test_spam_data['confidence_spam'] = confidence[:, 1] * 100
test_spam_data[['label', 'predicted', 'confidence_spam', 'email_text']]

,label,predicted,confidence_spam,email_text
0,1,1,92.106115,Congratulations! You have won a $1000 Amazon g...
1,0,0,7.086652,"Hi John, attached is the updated report you re..."
2,0,0,24.872124,Here are the notes from today's client meeting...
3,1,1,74.298960,"Winner, off, Congratulations, free"
4,0,0,20.338334,Meeting reminder: The project review is schedu...
5,1,0,35.128463,Act now! Lowest mortgage rates available. Pre-...
6,1,1,59.290079,Urgent! Your bank account has been suspended. ...
7,0,0,42.657091,Thank you for your purchase. Your order will b...
8,1,0,15.813559,Exclusive deal just for you. Get 90% off luxur...
9,1,1,62.376931,"Dear customer, your package delivery failed. C..."
